# 2. Автоматическая генерация и поиск признаков

Идея: связь × агрегат × окно. Человек задаёт операции и отношения; программа выбирает комбинации и уточняет окна. Это ограниченная грамматика и локальный поиск.

In [1]:
from pathlib import Path
import json, sys
import numpy as np
import pandas as pd
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "src/research/feature_automation").is_dir())
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
RUN_TRAINING = False
RESULT_DIR = ROOT / ("results/runs/notebook-reproduction" if RUN_TRAINING else "results/final")
DATA_DIR = ROOT / "data/processed"
read = lambda name: json.loads((RESULT_DIR / name).read_text())
print("Результаты фиксированного эксперимента; RUN_TRAINING =", RUN_TRAINING)


Результаты фиксированного эксперимента; RUN_TRAINING = False


In [2]:
if RUN_TRAINING:
    import subprocess
    subprocess.run([sys.executable, "-m", "research.feature_automation.experiment",
                    "--data-dir", str(DATA_DIR), "--out", str(RESULT_DIR), "--phase", "automatic"],
                   cwd=ROOT, check=True)
else:
    print("Этап automatic ранее выполнен CLI; ниже его журнал и результаты.")


Этап automatic ранее выполнен CLI; ниже его журнал и результаты.


In [3]:
from research.feature_automation.features import generated_features
generated = generated_features()
print("Сгенерировано:", len(generated))
print(pd.DataFrame([s.description() for s in generated]).to_string(index=False))

Сгенерировано: 102
   relation operation  days  months                       name
       user     count     7       0            user__count__7d
       user    unique     7       0           user__unique__7d
       user     count    14       0           user__count__14d
       user    unique    14       0          user__unique__14d
       user     count    30       0           user__count__30d
       user    unique    30       0          user__unique__30d
       user     count    60       0           user__count__60d
       user    unique    60       0          user__unique__60d
       user     count    90       0           user__count__90d
       user    unique    90       0          user__unique__90d
       user     count   180       0          user__count__180d
       user    unique   180       0         user__unique__180d
       user   recency   180       0        user__recency__180d
       user     trend    30       0           user__trend__30d
       item     count     7       0 

## Отбор

Forward selection по внутреннему Recall@5, без построчных перестановок. После отбора связей проверяется вклад окон и соседние значения. После этого настраивается бустинг.

In [4]:
a = read("automatic_stage.json")
print("Выбранные связи:", a["selected_families"])
print("Шаги:", json.dumps(a["selection_steps"], ensure_ascii=False, indent=2))
print("Уточнение:", json.dumps(a["window_refinement"], ensure_ascii=False, indent=2))
print("Добавлено к ручной базе:", a["combined_added_families"])

Выбранные связи: ['pair', 'item']
Шаги: [
  {
    "family": "pair",
    "previous_recall": null,
    "recall": 0.5342392715721592,
    "trial": 2
  },
  {
    "family": "item",
    "previous_recall": 0.5342392715721592,
    "recall": 0.5978963253027424,
    "trial": 7
  }
]
Уточнение: [
  {
    "family": "pair",
    "drop_importance": [
      [
        0.006206015464036874,
        7
      ],
      [
        0.0009921117207213337,
        14
      ],
      [
        -0.005393169664560049,
        30
      ],
      [
        -0.00211304218657149,
        60
      ],
      [
        -0.0007808701993193834,
        90
      ],
      [
        0.0072405201750120884,
        180
      ]
    ],
    "proposed_days": [
      135
    ],
    "attempts": [
      {
        "days": 135,
        "recall": 0.5858323939874742,
        "trial": 22
      }
    ],
    "accepted": null
  },
  {
    "family": "item",
    "drop_importance": [
      [
        0.002449967590341884,
        7
      ],
      [


In [5]:
for name, r in a["frozen"].items():
    print(name, "features:", len(r["features"]), "params:", r["params"])
    print("CV:", a["best"][name]["aggregate"], "validation:", r["validation"])
print("Исходная ручная модель сохранена:", a["manual_model_unchanged"])
print("Профиль:", a["timing"])

automatic_all features: 102 params: {'learning_rate': 0.05, 'max_iter': 100, 'max_leaf_nodes': 7, 'min_samples_leaf': 60, 'l2_regularization': 0.0}
CV: {'Recall@5': 0.6018798825951233, 'NDCG@5': 0.5146254167774481, 'Precision@5': 0.22647058823529412, 'LogLoss': 0.025555954543381526} validation: {'Recall@5': 0.6470223816377662, 'NDCG@5': 0.5603878564420564, 'Precision@5': 0.25696465696465703, 'Precision@5_all': 0.0412, 'rows': 150000, 'positives': 1068, 'eligible_cases': 481, 'all_cases': 3000, 'LogLoss': 0.02565492921887811, 'ROC-AUC': 0.9481392062043547}
automatic_selected features: 28 params: {'learning_rate': 0.1, 'max_iter': 100, 'max_leaf_nodes': 7, 'min_samples_leaf': 20, 'l2_regularization': 0.0}
CV: {'Recall@5': 0.5978963253027424, 'NDCG@5': 0.5139625936554749, 'Precision@5': 0.22513368983957216, 'LogLoss': 0.027686065928002036} validation: {'Recall@5': 0.6225058321212168, 'NDCG@5': 0.5455828030673003, 'Precision@5': 0.2444906444906445, 'Precision@5_all': 0.039200000000000006, 